<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [1]</a>'.</span>

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [1]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, BooleanType

offline_packages_path = "/home/jovyan/work/storage/packages"
if offline_packages_path not in sys.path:
    sys.path.insert(0, offline_packages_path)

internal_ivy_path = "/home/jovyan/.ivy2/jars/*"

# 1. بناء الجلسة
spark = SparkSession.builder \
    .appName("Bronze_To_Silver_Cleaner") \
    .config("spark.jars", internal_ivy_path) \
    .getOrCreate()

# 2. تعريف المخطط الصارم لقراءة الملفات البرونزية
bronze_schema = StructType([
    StructField("timestamp", StringType(), True),
    StructField("house_type", StringType(), True),
    StructField("currency", StringType(), True),
    StructField("zone", StringType(), True),
    StructField("device_id", StringType(), True),
    StructField("device_type", StringType(), True),
    StructField("is_room_occupied", BooleanType(), True), 
    StructField("power_consumption_watts", DoubleType(), True),
    StructField("status", StringType(), True)
])

try:
    print("📂 [السيلفر] جاري مراقبة الملفات الجديدة في الطبقة البرونزية...")
    
    # القراءة التزايدية من مجلد البرونز الخام
    bronze_stream = spark.readStream \
        .schema(bronze_schema) \
        .parquet("/home/jovyan/work/storage/raw_bronze_parquet")

    # 3. تطبيق قواعد التنظيف والتعقيم (تصفية الأخطاء والمفقودات)
    silver_cleaned_df = bronze_stream.filter(
        (col("power_consumption_watts").isNotNull()) &
        (col("power_consumption_watts") >= 0.0) &
        (col("power_consumption_watts") <= 3500.0) &
        ~((col("device_type") == "Lighting") & (col("power_consumption_watts") > 100.0))
    )

    silver_parquet_path = "/home/jovyan/work/storage/historical_parquet"
    checkpoint_silver = "/home/jovyan/work/storage/checkp_silver_clean"

    # 🛑 تم حذف سطر تصفير الـ Checkpoint التلقائي لضمان التتبع التزايدي الحقيقي للبيانات!

    print("💾 [السيلفر] جاري حقن البيانات المعقمة في المجلد الفضي الفعلي...")
    
    # الكتابة التزايدية إلى مجلد السيلفر
    query = silver_cleaned_df.writeStream \
        .format("parquet") \
        .outputMode("append") \
        .partitionBy("zone") \
        .option("path", silver_parquet_path) \
        .option("checkpointLocation", checkpoint_silver) \
        .trigger(availableNow=True) \
        .start()

    query.awaitTermination()
    print("✅ [السيلفر] تم تصفية البيانات بنجاح وأصبحت جاهزة للتحليل في الطبقة الذهبية!")

except Exception as e:
    print(f"❌ فشل خط معالجة السيلفر: {e}")
finally:
    # إغلاق الجلسة بأمان
    spark.stop()

ModuleNotFoundError: No module named 'pyspark'

import shutil

# 1. تصفير مجلد السيلفر والـ Checkpoint الخاص به ليعيد بناء التواريخ نظيفة تماماً
shutil.rmtree("/home/jovyan/work/storage/historical_parquet", ignore_errors=True)
shutil.rmtree("/home/jovyan/work/storage/checkp_silver_clean", ignore_errors=True)

# 2. تصفير الـ Checkpoint الخاص بكود الـ DWH الموحد ليعمل على قراءة كامل البيانات الفضية المعاد بناؤها
shutil.rmtree("/home/jovyan/work/storage/checkp_analytics_increment", ignore_errors=True)

print("🧹 تمت عملية التطهير بنجاح! جاهز الآن لتشغيل خط الأنابيب النظيف.")